# AFS Ingest

Upload PDFs to `@AFS_STAGE`, run `PARSE_DOCUMENT()` on each, and write extracted text to `COMMON.PDF_STAGING`.

**How to upload PDFs:**
- In the Snowflake UI, use the Files tab of `@AUDITED_FINANCIALS.COMMON.AFS_STAGE` to drag-and-drop PDF files.
- Or from SnowSQL / Snowflake CLI: `PUT file:///path/to/file.pdf @AUDITED_FINANCIALS.COMMON.AFS_STAGE`

Then run this notebook to extract and stage text for any PDFs not yet in `PDF_STAGING`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import importlib
import afs.pdf_ingest
importlib.reload(afs.pdf_ingest)
from afs.pdf_ingest import extract_and_stage
import json

In [ ]:
# List PDFs in stage
staged_files = session.sql(
    "LIST @AUDITED_FINANCIALS.COMMON.AFS_STAGE PATTERN='.*\\.pdf'"
).collect()

print(f'Files in stage: {len(staged_files)}')
for f in staged_files:
    print(' -', f[0])

In [ ]:
already_staged = {
    row[0] for row in session.sql(
        "SELECT FILENAME FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING"
    ).collect()
}

seen_md5 = set()
pending = []
skipped_dup = 0
for f in staged_files:
    full_path = f[0]   # e.g. afs_stage/medstar/uuid.pdf
    md5 = f[2]
    rel_path = '/'.join(full_path.split('/')[1:])  # strip stage name prefix
    bare_name = full_path.split('/')[-1]
    if bare_name in already_staged or rel_path in already_staged:
        continue
    if md5 in seen_md5:
        skipped_dup += 1
        continue
    pending.append(rel_path)
    seen_md5.add(md5)

print(f'Pending extraction: {len(pending)} ({skipped_dup} duplicate(s) skipped)')
for name in pending:
    print(' -', name)

In [ ]:
from datetime import datetime

results = []
for filename in pending:
    print(f'\nExtracting: {filename} ...', flush=True)
    try:
        row = extract_and_stage(session, filename)
        session.sql("""
            INSERT INTO AUDITED_FINANCIALS.COMMON.PDF_STAGING
              (FILENAME, FILING_ID, TOTAL_PAGES, PAGE_TEXTS, STATUS)
            SELECT ?, ?, ?, PARSE_JSON(?), 'pending'
        """, params=[
            row['filename'],
            row['filing_id'],
            row['total_pages'],
            json.dumps(row['page_texts']),
        ]).collect()
        results.append({'file': filename, 'pages': row['total_pages'], 'status': 'ok'})
        print(f'  -> {row["total_pages"]} pages, filing_id={row["filing_id"][:12]}...')
    except Exception as e:
        results.append({'file': filename, 'status': 'error', 'error': str(e)})
        print(f'  -> ERROR: {e}')

print(f'\nDone. {sum(1 for r in results if r["status"]=="ok")}/{len(results)} succeeded.')

In [ ]:
# Show pending filings ready to parse
import pandas as pd

df = session.sql("""
    SELECT STAGING_ID, FILENAME, TOTAL_PAGES, STATUS, EXTRACTED_AT
      FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING
     ORDER BY EXTRACTED_AT DESC
""").to_pandas()

print(f'Total in PDF_STAGING: {len(df)}')
df

In [ ]:
from snowflake.snowpark import Session as _S
session = _S.builder.getOrCreate()
session.sql('USE WAREHOUSE DATARISE_INT').collect()